In [16]:
import pandas as pd
import re
import nltk
import seaborn as sns
import matplotlib.pyplot as plt
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
import umap
import hdbscan
from scipy.stats import chi2_contingency
from src.fct_data import extract_texts_from_zips


## Extraction des données



Traitement pour les 3 années disponibles sur le `gitlab` 

In [18]:
#lecture des métadonnées
metadonnees = pd.read_csv("data/archelect_search.zip", compression="zip")

# Extraire tous les textes depuis tous les ZIPs
meta_et_texts = extract_texts_from_zips("data/text_files")

# Afficher le premier texte
print(meta_et_texts['text'].iloc[0])

# Afficher les 5 premiers documents
meta_et_texts.head()

/tmp/ipykernel_24969/688946421.py:2: DtypeWarning: Columns (0: departement-nom, 1: departement-insee, 2: identifiant de circonscription, 3: pdf, 4: suppleant-nom, 5: suppleant-prenom, 6: suppleant-sexe, 7: suppleant-age, 8: suppleant-age-calcule, 9: suppleant-age-tranche, 10: suppleant-profession, 11: suppleant-mandat-en-cours, 12: suppleant-mandat-passe, 13: suppleant-associations, 14: suppleant-autres-statuts, 15: suppleant-soutien, 16: suppleant-liste, 17: suppleant-decorations) have mixed types. Specify dtype option on import or set low_memory=False.
  metadonnees = pd.read_csv("data/archelect_search.zip", compression="zip")


Traitement de data/text_files/1981/legislatives.zip


Traitement de data/text_files/1993/legislatives.zip
Traitement de data/text_files/1993/presidentielle.zip
Traitement de data/text_files/1988/legislatives.zip
Nombre total de documents extraits : 12746
ELECTIONS LEGISLATIVES DES 14 ET 21 JUIN 1981
Electrices, Electeurs
de la 3eme Circonscription
Une ère nouvelle vient de commencer, un nouveau pouvoir politi- que a été mis en place. Nous y avons contribué. Il faut une autre politique, il faut des hommes nouveaux.
Les 14 e: 21 Juin vous au- rez à élire un Député à l'Assemblée Na- tionale Française. Le vote est important compte tenu des possibilités nouvelles qui nous sont offertes avec l'arrivée de la Gauche au pouvoir. Beaucoup d'entre vous ont été odieusement trompés par la presse écrite et audio-visuelle aux ordres de l'ancien pouvoir qui l'avait confis- quée. Chaque fois qu'il s'agissait de choi- sir un représentant, le pouvoir vous dési- gnait le plus mauvais sous le prétexte fallacieux que le statut politique de la Guadeloupe était 

,id,text,zip_path
0,EL137_L_1981_06_971_03_1_PF_04,ELECTIONS LEGISLATIVES DES 14 ET 21 JUIN 1981\...,data/text_files/1981/legislatives.zip
1,EL134_L_1981_06_014_01_1_PF_06,Département du Calvados\nRépublique Française\...,data/text_files/1981/legislatives.zip
2,EL134_L_1981_06_025_01_1_PF_09,"Sciences Po / fonds CEVIPOF\nElectrices, Elect...",data/text_files/1981/legislatives.zip
3,EL135_L_1981_06_061_01_1_PF_02,· République Française Élections législatives ...,data/text_files/1981/legislatives.zip
4,EL137_L_1981_06_076_01_2_PF_02,Sciences Po / fonds CEVIPOF\nRÉPUBLIQUE FRANÇA...,data/text_files/1981/legislatives.zip


## Pre-traitement des données

In [19]:
nltk.download('stopwords')
stop_words = set(stopwords.words('french'))

[nltk_data] Downloading package stopwords to /home/onyxia/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [7]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    words = [w for w in text.split() if w not in stop_words]
    return ' '.join(words)

meta_et_texts['texte_clean'] = meta_et_texts['text'].apply(clean_text)
meta_et_texts.drop_duplicates(inplace=True)
print(f"Dataset chargé : {meta_et_texts.shape[0]} lignes")

Dataset chargé : 3182 lignes


## Représentation textuelle

In [9]:
model = SentenceTransformer('camembert-base')
embeddings = model.encode(meta_et_texts['texte_clean'].tolist(), batch_size=16, show_progress_bar=True)
print(f"Embeddings : {embeddings.shape}")

No sentence-transformers model found with name camembert-base. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 736.10it/s, Materializing param=pooler.dense.weight]                               
CamembertModel LOAD REPORT from: camembert-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.bias              | UNEXPECTED |  | 
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 199/199 [13:14<00:00,  3.99s/it]

Embeddings : (3182, 768)


## Classification par thème

In [10]:
umap_embeddings = umap.UMAP(
    n_neighbors=15, min_dist=0.0, n_components=5, random_state=42
).fit_transform(embeddings)
print(f"UMAP embeddings shape: {umap_embeddings.shape}")

/opt/python/lib/python3.13/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


UMAP embeddings shape: (3182, 5)


In [11]:
print("Clustering avec HDBSCAN...")
clusterer = hdbscan.HDBSCAN(min_cluster_size=10)
meta_et_texts['theme_cluster'] = clusterer.fit_predict(umap_embeddings)
print(f"Clusters trouvés : {meta_et_texts['theme_cluster'].nunique()} (le -1 correspond aux outliers)")


Clustering avec HDBSCAN...
Clusters trouvés : 17 (le -1 correspond aux outliers)


## Croisement avec les professions des candidats

On fait un peu de visualisation

In [12]:
plt.figure(figsize=(12,6))
sns.countplot(x='theme_cluster', hue='profession', data=meta_et_texts)
plt.title("Répartition des thèmes abordés par métier")
plt.xlabel("Thème")
plt.ylabel("Nombre de professions de foi")
plt.legend(title="Métier")
plt.show()

ValueError: Could not interpret value `profession` for `hue`. An entry with this name does not appear in `data`.

<Figure size 1200x600 with 0 Axes>

## Analyses statistiques diverses

In [ ]:
contingency_table = pd.crosstab(meta_et_texts['profession'], meta_et_texts['theme_cluster'])
chi2, p, dof, expected = chi2_contingency(contingency_table)
print(f"Chi2 = {chi2:.2f}, p-value = {p:.4f}")